# <font color="#418FDE" size="10" uppercase>**C: SimCLR Contrastive Experiments**</font>
----

> Last update: 20240312

By the end of this lecture, you will be able to:
* Create SimCLR experiments without pre-trained weights.
* Create SimCLR experiments with pre-trained weights.

## **1. SimCLR Experiment - without Pre-Trained Weights**

> In this experiment, the focus is on training models <u>***without***</u> the advantage of pre-trained parameters due to the unavailability of a network trained on a similar data distribution and using the SimCLR technique. Besides, we assume we have limited labeled data from the CIFAR-10 dataset, specifically using only 1000 labeled images. This way, we simulate a real-world labeling challenge scenario.

> The approach involves developing a SimCLR unsupervised contrastive pretext model (`model_prx`) utilizing all training inputs, which is then transfer-learned and fine-tuned on the 1000 labeled images for the downstream task (`model_dwm`). This fine-tuned model is used to label testing images.

> For comparative analysis, a fully supervised model (`model_fsp`) is also trained solely on the 1000 labeled images. The key comparison is between the accuracies of `model_fsp` and `model_dwm` on the testing data, highlighting the effectiveness of the contrastive pretext strategy with limited labeled data.

In [ ]:
#@title Imports & Hyper-Parameters
'''
tf.__version__: 2.15.0
Runtime: GPU $$$ ~6000 sec with T4 GPUs
After the first run, restart the runtime if you want to run this cell for the
second time to prevent memory errors.

Abbreviations:
    acc: accuracy
    datain: input data
    dataou: output data
    fsp: fully supervised learning
    prx: pretext
    dwm: downstream
    trf: transfer learning
    fnt: fine-tuning
    lr: learning rate
    tf: tensorflow
    tr: training
    te: testing

Remember: we want to practice Self-Supervised Learning (prx + dwm) and compare
it with fsp scenario. So, we take some measures to make sure we have a fair comparison.
'''

# Reset the memory to make sure we don't have garbage in our limited free memory!
%reset -f


# Import necessary libraries
import os
import tensorflow as tf
## Disable TensorFlow warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # 0 = all messages are logged (default behavior)
                                          # 1 = INFO messages are not printed
                                          # 2 = INFO and WARNING messages are not printed
                                          # 3 = INFO, WARNING, and ERROR messages are not printed
import pandas as pd
from tqdm import tqdm
import time
import copy
import warnings
warnings.filterwarnings('ignore')

'''
Hyper-parameters
'''

num_labeled  = 1000

## learning rates
lr_fsp     = 0.001
lr_prx     = 0.001
lr_dwm_trf = 0.01
lr_dwm_fnt = 0.001

## batch sizes
batch_fsp = 128
batch_prx = 64 # Since we have two augmentations in SimCLR, we get to 2 * 64 = 128 concatenated batch
batch_dwm = 128

## epochs: We keep epoch_fsp  = epoch_dwm_trf + epoch_dwm_fnt
## for a fair comparison.
epoch_fsp     = 25
epoch_prx     = 25
epoch_dwm_trf = 15
epoch_dwm_fnt = 10

## Image resolution for upscaling CIFAR10 data
global res
res = 128

## Print the version of TensorFlow
print(tf.__version__)

In [ ]:
#@title Data Preparation

# Load CIFAR-10 dataset, splitting into training and testing sets
(datain_tr, dataou_tr), (datain_te, dataou_te) = tf.keras.datasets.cifar10.load_data()

# Normalize training and testing input data from integers (0-255) to floats (0-1)
datain_tr = datain_tr / 255.0
datain_te = datain_te / 255.0

# Convert training and testing output labels to one-hot encoding
dataou_tr = tf.keras.utils.to_categorical(dataou_tr)
dataou_te = tf.keras.utils.to_categorical(dataou_te)

# Print the shapes of the input and output data sets for both training and testing
print('Shape of datain_tr: {}'.format(datain_tr.shape))
print('Shape of datain_te: {}'.format(datain_te.shape))
print('Shape of dataou_tr: {}'.format(dataou_tr.shape))
print('Shape of dataou_te: {}'.format(dataou_te.shape))

# Data Augmentation

# Define two data augmentations.
fun_augment_01 = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.2),
])

fun_augment_02 = tf.keras.Sequential([
    tf.keras.layers.RandomCrop(height = int(res/2), width = int(res/2)),
    tf.keras.layers.Resizing(height = res, width  = res)
])


# We only have input data for prx; we don't use any prx validation data in this experiment
datain_tr_prx = copy.deepcopy(datain_tr)

# Limit the labeled training data

# Randomly select a subset of the training data to simulate a limitted labeled repository scenario
index_tr = tf.experimental.numpy.random.randint(0, datain_tr.shape[0], num_labeled, dtype=tf.int32)

# Gather the selected subset of labeled training data
datain_tr_labeled = tf.gather(datain_tr, index_tr, axis=0)
dataou_tr_labeled = tf.gather(dataou_tr, index_tr, axis=0)

# Deep copy the labeled training data for fsp training (for code readability)
datain_tr_fsp = copy.deepcopy(datain_tr_labeled)
dataou_tr_fsp = copy.deepcopy(dataou_tr_labeled)

# Another deep copy for dwm training (for code readability)
datain_tr_dwm = copy.deepcopy(datain_tr_labeled)
dataou_tr_dwm = copy.deepcopy(dataou_tr_labeled)

# Define an upscaling function to be later used in the tf dataset API
def fun_upscale(image, label=None):
    # Resize the image to 224x224 pixels.
    image_resized = tf.image.resize_with_pad(image, res, res, method = tf.image.ResizeMethod.BILINEAR)
    return image_resized, label

# Prepare TensorFlow datasets for different training, validation, and testing purposes
# with batch sizes specified by variables and prefetch for performance optimization.
buffer_size    = datain_tr_fsp.shape[0]
dataset_tr_fsp = tf.data.Dataset.from_tensor_slices((datain_tr_fsp, dataou_tr_fsp)).map(fun_upscale).shuffle(buffer_size, reshuffle_each_iteration=True).batch(batch_fsp).prefetch(tf.data.experimental.AUTOTUNE)
dataset_tr_prx = tf.data.Dataset.from_tensor_slices((datain_tr_prx)).map(fun_upscale).shuffle(buffer_size, reshuffle_each_iteration=True).batch(batch_prx).prefetch(tf.data.experimental.AUTOTUNE)
dataset_tr_dwm = tf.data.Dataset.from_tensor_slices((datain_tr_dwm, dataou_tr_dwm)).map(fun_upscale).shuffle(buffer_size, reshuffle_each_iteration=True).batch(batch_dwm).prefetch(tf.data.experimental.AUTOTUNE)
dataset_te     = tf.data.Dataset.from_tensor_slices((datain_te, dataou_te)).map(fun_upscale).batch(128).prefetch(tf.data.experimental.AUTOTUNE)


In [ ]:
#@title Create Models FSP and PRX

# Load the base model (DenseNet121 in this example) with ImageNet weights
model_base = tf.keras.applications.DenseNet121(include_top=False,
                                               weights=None,
                                               input_shape=(res, res, 3))

# We clone model_base to create model_base_fsp and model_base_prx model.
# P.S. We will clone the model_dws after the pretext task using model_prx.
model_base_fsp = tf.keras.models.clone_model(model_base)
model_base_prx = tf.keras.models.clone_model(model_base)

# We set the parameters of model_base_fsp and model_base_prx to be the same as the
# randomly generated parameters of model_base to have both models' initial
# weights (i.e., start-point) the same for a fair comparison.
model_base_fsp.set_weights(model_base.get_weights())
model_base_prx.set_weights(model_base.get_weights())

print('Example of weights in the 3rd  layer of model_base_fsp:', model_base_fsp.layers[2].weights[0][0][0][0][:2])
print('Example of weights in the 3rd  layer of model_base_prx:', model_base_prx.layers[2].weights[0][0][0][0][:2])

# SimCLR projector
projector_input_shape = int(tf.math.reduce_prod(model_base_prx.layers[-1].output_shape[1:]))
model_projector       = tf.keras.Sequential([tf.keras.Input(shape=(projector_input_shape)),
                                             tf.keras.layers.Dense(512, activation = 'relu'),
                                             tf.keras.layers.Dense(128, activation = 'relu')])


# Now we create the model_fsp and model_prx.
inputs_fsp  = tf.keras.Input(shape=(res, res, 3))
x_fsp       = model_base_fsp(inputs_fsp)
x_fsp       = tf.keras.layers.Flatten()(x_fsp)
x_fsp       = tf.keras.layers.BatchNormalization()(x_fsp)
outputs_fsp = tf.keras.layers.Dense(10, activation='softmax')(x_fsp) # ten outputs
model_fsp   = tf.keras.Model(inputs_fsp, outputs_fsp)


inputs_prx  = tf.keras.Input(shape=(res, res, 3))
x_prx       = model_base_prx(inputs_prx)
x_prx       = tf.keras.layers.Flatten()(x_prx)
x_prx       = tf.keras.layers.BatchNormalization()(x_prx)
outputs_prx = model_projector(x_prx) # there are no actual outputs but the last layer of projector
model_prx   = tf.keras.Model(inputs_prx, outputs_prx)


In [ ]:
#@title Some Useful Functions
'''
Abbreviations:
    datain: Input data
    ind   : Index
    tf    : TensorFlow
'''

# Function definition for training a model using the SimCLR approach
def fun_train_simclr(model, dataset, fun_augment_01, fun_augment_02,
                     epochs=100, verbose=1, patience=3, learning_rate=0.001):

    # Determine the output size of the model's last layer
    z_size = model.layers[-1].weights[-1].shape[0]

    # Initialize a list to keep track of the loss values for each epoch
    loss = []

    # Initialize the optimizer with the specified learning rate
    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)

    # Loop through each epoch
    for epoch in range(epochs):
        # Initialize a variable to keep track of the running loss for the current epoch
        loss_running = 0

        # Initialize a progress bar using tqdm to visualize training progress
        pbar = tqdm(dataset,
                    desc  = f"SimCLR Training: {epoch + 1:03d}/{epochs}", # Description shown in the progress bar
                    ncols = 125, # Width of the progress bar
                    leave = True) # Whether the progress bar should remain after completion

        # Iterate over each batch in the dataset
        for batch_num, batch in enumerate(pbar, 1):
            if isinstance(batch, tuple):
                batch_in = batch[0] # Get the input data from the batch
            else:
                batch_in = batch

            # Apply two different augmentations to the input data
            x_tilda_01 = fun_augment_01(batch_in)
            x_tilda_02 = fun_augment_02(batch_in)

            # Concatenate the augmented data along the batch axis and reshape it for the model input
            x_tilda = tf.reshape(tf.concat([x_tilda_01, x_tilda_02], axis=0),
                                (x_tilda_01.shape[0] * 2, # Corrected to ensure the correct shape for concatenation
                                 x_tilda_01.shape[1], x_tilda_01.shape[2],
                                 x_tilda_01.shape[3]))

            # Create a dummy output tensor of the same size as the model's output
            z_real = tf.random.uniform((x_tilda.shape[0], z_size))

            # Open a GradientTape to record the operations for automatic differentiation
            with tf.GradientTape() as tape:
                # Pass the concatenated and augmented inputs through the model
                z_estimate = model(x_tilda, training=True)

                # Calculate the batch loss using the custom SimCLR loss function
                loss_batch = fun_simclr_loss(z_real, z_estimate)

            # Calculate the gradients of the loss with respect to the model's trainable variables
            gradients = tape.gradient(loss_batch, model.trainable_variables)

            # Apply the calculated gradients to the model's variables to minimize the loss
            optimizer.apply_gradients(zip(gradients, model.trainable_variables))

            # Update the running loss for the epoch
            loss_running += loss_batch.numpy()  # Ensure loss is a scalar by calling .numpy()

            # If verbose, update the progress bar with the current average loss
            if verbose:
                pbar.set_postfix(loss = f"{loss_running/batch_num:.6f}")

        # Append the average loss for this epoch to the loss history
        loss.append(loss_running / batch_num)

        # Check for early stopping: if there's no improvement in loss for a specified number of epochs, stop training
        if epoch >= patience:
            if loss[-1] > min(loss[:-patience]):
                print("Early stopping due to no improvement in loss.")
                break  # Exit the training loop

    # Return the trained model and the history of loss values
    return model, loss

#SimCLR loss function
@tf.function
def fun_simclr_loss(z_real, z_estimate):
    # The z_real parameter is not used since SimCLR is an unsupervised model.
    # This dummy variable is discarded.
    del z_real

    # Temperature parameter is set to 0.1. This hyperparameter can be tuned
    # for specific problems to control the smoothness of the output distribution.
    toe = .1

    # Calculate the number of projections, which is twice the batch size (2N).
    num = z_estimate.shape[0]

    # Generate two sets of indices for all possible pair combinations within the batch.
    # ind0 is a temporary variable holding the repeated range for generating indices.
    ind0 = tf.repeat(tf.expand_dims(tf.range(0, num), axis=0), num, axis=0)
    # Flatten ind0 to create a single list of indices for the first element of pairs.
    ind1 = tf.reshape(ind0, (num**2, 1))[:, 0]
    # Flatten the transpose of ind0 to create a single list of indices for the second element of pairs.
    ind2 = tf.reshape(tf.transpose(ind0), (num**2, 1))[:, 0]

    # The temporary index tensor is no longer needed after use.
    del ind0

    # Select the projections based on the first set of indices to form the first elements of pairs.
    vector_1 = tf.gather(z_estimate, ind1, axis=0)
    # The first set of indices is no longer needed after use.
    del ind1

    # Select the projections based on the second set of indices to form the second elements of pairs.
    vector_2 = tf.gather(z_estimate, ind2, axis=0)
    # The second set of indices is no longer needed after use.
    del ind2

    # Calculate the cosine similarity between each pair of projections and negate it to prepare for loss calculation.
    s = -tf.reshape(tf.keras.losses.cosine_similarity(vector_1, vector_2, axis=1), (num, num))

    # The vector tensors are no longer needed after computing similarities.
    del vector_1
    del vector_2

    # Calculate the nominator of the contrastive loss function for each pair.
    nom = tf.exp(s / toe)

    # Calculate the denominator of the contrastive loss function by excluding the self-similarity term.
    x1 = tf.exp(s / toe)
    x2 = 1 - tf.eye(num, dtype=tf.float32)
    denom = tf.repeat(tf.expand_dims(tf.math.reduce_sum(x1 * x2, axis=1), axis=1), num, axis=1)

    # The intermediate tensors used for the denominator are no longer needed.
    del s
    del x1
    del x2

    # Calculate the loss `l(i, j)` for each pair of projections using the computed nominator and denominator.
    l = -tf.math.log(nom / denom)

    # Cleanup intermediate tensors to save memory.
    del nom
    del denom

    # Prepare indices to extract the diagonal elements from the loss matrix, which correspond to the actual pair losses.
    ind_2k0 = tf.range(0, num, 2, dtype=tf.int32)  # Even indices
    ind_2k1 = tf.range(1, num, 2, dtype=tf.int32)  # Odd indices

    # Extract and sum the losses for the actual pairs using the prepared indices.
    loss_mat1_1 = tf.gather(l, ind_2k0, axis=0)
    loss_mat1_2 = tf.gather(loss_mat1_1, ind_2k1, axis=1)
    loss_mat1 = tf.linalg.diag_part(loss_mat1_2)

    loss_mat2_1 = tf.gather(l, ind_2k1, axis=0)
    loss_mat2_2 = tf.gather(loss_mat2_1, ind_2k0, axis=1)
    loss_mat2 = tf.linalg.diag_part(loss_mat2_2)

    # After extracting the necessary elements, the large loss tensor can be discarded.
    del l

    # Combine the individual loss components into a single tensor.
    loss_mat = loss_mat1 + loss_mat2

    # Compute the final loss by taking the sum over all individual losses and normalizing by the number of pairs.
    L = tf.math.reduce_sum(loss_mat) / num


    # Return the final computed loss.
    return L

# Function to compile a model with specified learning rate
def fun_model_compile(model, learning_rate, loss = 'categorical_crossentropy', metrics = [['accuracy']] ):
    # Compiles the model with Adam optimizer, categorical crossentropy as loss, and tracks accuracy
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  loss=loss,
                  metrics=metrics)
    return model

# Function to print the summary of the model
def fun_model_summary(model):
    model.summary()  # Prints the summary of the model
    print('\n')  # Prints a newline for readability

# Function to define callbacks for training
def fun_model_callbacks():
    # Early stopping callback to stop training when val_loss doesn't improve
    cb_early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    # Reduce learning rate callback when val_loss plateaus
    cb_reduce_lr      = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=0.00001)
    return [cb_early_stopping, cb_reduce_lr]

# Function to train the models other than SimCLR with given dataset, epochs, and callbacks
def fun_model_train(model, dataset_tr, dataset_vl, epochs, callbacks):
    model.fit(dataset_tr,  # Training data
              epochs          = epochs,  # Number of epochs to train for
              verbose         = 1,  # Verbosity mode
              shuffle         = True,  # Whether to shuffle the training data
              validation_data = dataset_vl,  # Validation data
              callbacks       = callbacks)  # Callbacks for training
    return model

# Function to evaluate the model with a given dataset
def fun_model_evaluate(model, data):
    _, acc = model.evaluate(data, verbose = 3)
    return [acc]

# Function to compare the weights of base and trained models in fsp and prx tasks
def fun_control(model_base_fsp, model_base_prx, model_fsp, model_prx):
    print('\n')
    # Print examples of weights from the 3rd layer of fully supervised base model
    print('Example of weights in the 3rd  layer of model_base_fsp: ', model_base_fsp.layers[2].weights[0][0][0][0][:2])
    # Print examples of weights from the 3rd layer of pretext base model
    print('Example of weights in the 3rd  layer of model_base_prx: ', model_base_prx.layers[2].weights[0][0][0][0][:2])
    # Print examples of weights from the last layer of fully supervised trained model
    print('Example of weights in the last layer of model_fsp     : ', model_fsp.layers[-1].weights[0][:1][0][:2])
    # Print examples of weights from the last layer of pretext trained model
    print('Example of weights in the last layer of model_prx     : ', model_prx.layers[-1].weights[0][:1][0][:2])


In [ ]:
#@title Training Models
# Capture the start time
t0 = time.time()

# Initiate a result dictionary
results = {}

'''
Training model_fsp.
'''

print('\nTraining model_fsp\n')

# No pre-trained weights, no freezing!
model_base_fsp.trainable = True

# No pre-trained weights, no freezing!
model_fsp.layers[-2]     = True

# Compile the model with specified learning rate for transfer learning phase.
model_fsp = fun_model_compile(model_fsp, lr_fsp)

# Store the initial parameters of the last layer of model_fsp for comparison later.
outputs_fsp_initial_parameters = copy.deepcopy(model_fsp.layers[-1].weights)

# Display the model's structure.
fun_model_summary(model_fsp)

# Prepare callbacks for early stopping and learning rate adjustment.
callbacks = fun_model_callbacks()

# Start training the model with specified datasets and number of epochs.
model_fsp = fun_model_train(model_fsp, dataset_tr_fsp, dataset_te, epoch_fsp, callbacks)

# Evaluate the model with a specified dataset.
results ['fsp_trf'] = fun_model_evaluate(model_fsp, dataset_te)

# Check and display changes in parameters after training.
fun_control(model_base_fsp, model_base_prx, model_fsp, model_prx)



'''
Training model_prx.
'''

print('\nTraining model_prx\n')

# No pre-trained weights, no freezing!
model_base_prx.trainable = True

# No pre-trained weights, no freezing!
model_prx.layers[-2]     = True

# Show the model's structure.
fun_model_summary(model_prx)

# Begin training with specified configurations.
model_prx, _ = fun_train_simclr(model_prx,
                                dataset_tr_prx,
                                fun_augment_01,
                                fun_augment_02,
                                epochs     = epoch_prx,
                                verbose    = 1,
                                patience   = 1,
                                learning_rate = lr_prx)

# Evaluate changes in parameters after training.
fun_control(model_base_fsp, model_base_prx, model_fsp, model_prx)


'''
Construct and prepare model_dwm for transfer learning.
'''
# Constructing model_dwm utilizing the pre-trained model_base_prx, incorporating
# a dense layer similar to that in model_fsp.
inputs_dwm = tf.keras.Input(shape=(res, res, 3))
x_dwm = model_base_prx(inputs_dwm)  # Use the trained base model.
x_dwm = tf.keras.layers.Flatten()(x_dwm)
x_dwm = tf.keras.layers.BatchNormalization()(x_dwm)
outputs_dwm = tf.keras.layers.Dense(10, activation='softmax')(x_dwm)  # Set for 10 output classes.
model_dwm = tf.keras.Model(inputs_dwm, outputs_dwm)


# For equitable comparison between fsp and dwm, we align the parameters of dwm's
# final dense layer with outputs_fsp_initial_parameters.
# Additionally, we transfer and freeze the pre-trained weights of the preceding
# dense layer from prx into dwm's structure to leverage their learned
# representations.
# Set the initial parameters for fair comparison and incorporate pre-trained layers.
model_dwm.layers[-1].set_weights(outputs_fsp_initial_parameters)  # Match initial output layer parameters.
model_dwm.layers[-2].set_weights(model_prx.layers[-2].weights)  # Use pre-trained intermediate layer.

'''
Transfer learning for model_dwm.
'''

print('\nInitializing transfer learning for model_dwm\n')

# Freeze base and selected layers for the initial phase.
model_base_prx.trainable = False

# Freeze the batch norm layer to have its initial parameters (zeros and ones) intact.
model_dwm.layers[-2]     = False

# Compile the model for this phase.
model_dwm = fun_model_compile(model_dwm, lr_dwm_trf)

# Display the model structure.
fun_model_summary(model_dwm)

# Set callbacks for training.
callbacks = fun_model_callbacks()

# Begin training the model.
model_dwm = fun_model_train(model_dwm, dataset_tr_dwm, dataset_te, epoch_dwm_trf, callbacks)

# Evaluate the model with a specified dataset.
results ['dwm_trf'] = fun_model_evaluate(model_dwm, dataset_te)

# Check parameter changes after training.
fun_control(model_base_fsp, model_base_prx, model_fsp, model_prx)



'''
Fine-tuning for model_dwm.
'''

print('\nStarting fine-tuning for model_dwm\n')

# Unfreeze layers for fine-tuning.
model_base_prx.trainable = True

# Unfreeze the batch norm layer.
model_dwm.layers[-2]     = True

# Recompile with adjusted learning rate for fine-tuning.
model_dwm = fun_model_compile(model_dwm, lr_dwm_fnt)

# Display the model's updated structure.
fun_model_summary(model_dwm)

# Prepare callbacks for this phase.
callbacks = fun_model_callbacks()

# Proceed to fine-tune the model.
model_dwm = fun_model_train(model_dwm, dataset_tr_dwm, dataset_te, epoch_dwm_fnt, callbacks)

# Evaluate the model with a specified dataset.
results ['dwm_fnt'] = fun_model_evaluate(model_dwm, dataset_te)

# Evaluate parameter changes post fine-tuning.
fun_control(model_base_fsp, model_base_prx, model_fsp, model_prx)

# Display the results.
print('---- Accuracies ----')
print(pd.DataFrame(results).head())
print('\n')
print("Time duration of experiment (sec): ", time.time() - t0)

# Auto runtime disconnection to save CPU/GPU/TPU allocations
from google.colab import runtime
import time
time.sleep(10)
runtime.unassign()


## **2. SimCLR Experiment - with Pre-Trained Weights**

> In this experiment, the focus is on training models <u>***with***</u> the advantage of pre-trained parameters due to the availability of a network trained on a similar data distribution and using the SimCLR technique. Besides, we assume we have limited labeled data from the CIFAR-10 dataset, specifically using only 1000 labeled images. This way, we simulate a real-world labeling challenge scenario.

> The approach involves developing a SimCLR unsupervised contrastive pretext model (`model_prx`) utilizing all training inputs, which is then transfer-learned and fine-tuned on the 1000 labeled images for the downstream task (`model_dwm`). This fine-tuned model is used to label testing images.

> For comparative analysis, a fully supervised model (`model_fsp`) is also trained solely on the 1000 labeled images. The key comparison is between the accuracies of `model_fsp` and `model_dwm` on the testing data, highlighting the effectiveness of the contrastive pretext strategy with limited labeled data.

In [ ]:
#@title Imports & Hyper-Parameters
'''
tf.__version__: 2.15.0
Runtime: GPU $$$ ~6000 sec with T4 GPUs
After the first run, restart the runtime if you want to run this cell for the
second time to prevent memory errors.

Abbreviations:
    acc: accuracy
    datain: input data
    dataou: output data
    fsp: fully supervised learning
    prx: pretext
    dwm: downstream
    trf: transfer learning
    fnt: fine-tuning
    lr: learning rate
    tf: tensorflow
    tr: training
    te: testing

Remember: we want to practice Self-Supervised Learning (prx + dwm) and compare
it with fsp scenario. So, we take some measures to make sure we have a fair comparison.
'''

# Reset the memory to make sure we don't have garbage in our limited free memory!
%reset -f


# Import necessary libraries
import os
import tensorflow as tf
## Disable TensorFlow warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # 0 = all messages are logged (default behavior)
                                          # 1 = INFO messages are not printed
                                          # 2 = INFO and WARNING messages are not printed
                                          # 3 = INFO, WARNING, and ERROR messages are not printed
import pandas as pd
from tqdm import tqdm
import time
import copy
import warnings
warnings.filterwarnings('ignore')

# Import necessary libraries
import tensorflow as tf
import pandas as pd
import time
import copy

'''
Hyper-parameters
'''

num_labeled  = 1000

## learning rates
lr_fsp_trf = 0.01
lr_fsp_fnt = 0.0001
lr_prx_trf = 0.01
lr_prx_fnt = 0.00001
lr_dwm_trf = 0.01
lr_dwm_fnt = 0.0001

## batch sizes
batch_fsp = 64
batch_prx = 64
batch_dwm = 64

## epochs: We keep epoch_fsp_trf + epoch_fsp_fnt = epoch_dwm_trf + epoch_dwm_fnt
## for a fair comparison.
epoch_fsp_trf = 15
epoch_fsp_fnt = 10
epoch_prx_trf = 15
epoch_prx_fnt = 10
epoch_dwm_trf = 15
epoch_dwm_fnt = 10

## Image resolution for upscaling CIFAR10 data
global res
res = 128

## Print the version of TensorFlow
print(tf.__version__)

In [ ]:
#@title Data Preparation

# Load CIFAR-10 dataset, splitting into training and testing sets
(datain_tr, dataou_tr), (datain_te, dataou_te) = tf.keras.datasets.cifar10.load_data()

# Normalize training and testing input data from integers (0-255) to floats (0-1)
datain_tr = datain_tr / 255.0
datain_te = datain_te / 255.0

# Convert training and testing output labels to one-hot encoding
dataou_tr = tf.keras.utils.to_categorical(dataou_tr)
dataou_te = tf.keras.utils.to_categorical(dataou_te)

# Print the shapes of the input and output data sets for both training and testing
print('Shape of datain_tr: {}'.format(datain_tr.shape))
print('Shape of datain_te: {}'.format(datain_te.shape))
print('Shape of dataou_tr: {}'.format(dataou_tr.shape))
print('Shape of dataou_te: {}'.format(dataou_te.shape))

# Data Augmentation

# Define two data augmentations.
fun_augment_01 = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.2),
])

fun_augment_02 = tf.keras.Sequential([
    tf.keras.layers.RandomCrop(height = int(res/2), width = int(res/2)),
    tf.keras.layers.Resizing(height = res, width  = res)
])


# We only have input data for prx; we don't use any prx validation data in this experiment
datain_tr_prx = copy.deepcopy(datain_tr)

# Limit the labeled training data

# Randomly select a subset of the training data to simulate a limitted labeled repository scenario
index_tr = tf.experimental.numpy.random.randint(0, datain_tr.shape[0], num_labeled, dtype=tf.int32)

# Gather the selected subset of labeled training data
datain_tr_labeled = tf.gather(datain_tr, index_tr, axis=0)
dataou_tr_labeled = tf.gather(dataou_tr, index_tr, axis=0)

# Deep copy the labeled training data for fsp training (for code readability)
datain_tr_fsp = copy.deepcopy(datain_tr_labeled)
dataou_tr_fsp = copy.deepcopy(dataou_tr_labeled)

# Another deep copy for dwm training (for code readability)
datain_tr_dwm = copy.deepcopy(datain_tr_labeled)
dataou_tr_dwm = copy.deepcopy(dataou_tr_labeled)

# Define an upscaling function to be later used in the tf dataset API
def fun_upscale(image, label=None):
    # Resize the image to 224x224 pixels.
    image_resized = tf.image.resize_with_pad(image, res, res, method = tf.image.ResizeMethod.BILINEAR)
    return image_resized, label

# Prepare TensorFlow datasets for different training, validation, and testing purposes
# with batch sizes specified by variables and prefetch for performance optimization.
buffer_size    = datain_tr_fsp.shape[0]
dataset_tr_fsp = tf.data.Dataset.from_tensor_slices((datain_tr_fsp, dataou_tr_fsp)).map(fun_upscale).shuffle(buffer_size, reshuffle_each_iteration=True).batch(batch_fsp).prefetch(tf.data.experimental.AUTOTUNE)
dataset_tr_prx = tf.data.Dataset.from_tensor_slices((datain_tr_prx)).map(fun_upscale).shuffle(buffer_size, reshuffle_each_iteration=True).batch(batch_prx).prefetch(tf.data.experimental.AUTOTUNE)
dataset_tr_dwm = tf.data.Dataset.from_tensor_slices((datain_tr_dwm, dataou_tr_dwm)).map(fun_upscale).shuffle(buffer_size, reshuffle_each_iteration=True).batch(batch_dwm).prefetch(tf.data.experimental.AUTOTUNE)
dataset_te     = tf.data.Dataset.from_tensor_slices((datain_te, dataou_te)).map(fun_upscale).batch(128).prefetch(tf.data.experimental.AUTOTUNE)


In [ ]:
#@title Create Models FSP and PRX

# Load the base model (DenseNet121 in this example) with ImageNet weights
model_base = tf.keras.applications.DenseNet121(include_top=False,
                                               weights='imagenet',
                                               input_shape=(res, res, 3))

# We clone model_base to create model_base_fsp and model_base_prx model.
# P.S. We will clone the model_dws after the pretext task using model_prx.
model_base_fsp = tf.keras.models.clone_model(model_base)
model_base_prx = tf.keras.models.clone_model(model_base)

# We set the parameters of model_base_fsp and model_base_prx to be the same as the
# randomly generated parameters of model_base to have both models' initial
# weights (i.e., start-point) the same for a fair comparison.
model_base_fsp.set_weights(model_base.get_weights())
model_base_prx.set_weights(model_base.get_weights())

print('Example of weights in the 3rd  layer of model_base_fsp:', model_base_fsp.layers[2].weights[0][0][0][0][:2])
print('Example of weights in the 3rd  layer of model_base_prx:', model_base_prx.layers[2].weights[0][0][0][0][:2])

# SimCLR projector
projector_input_shape = int(tf.math.reduce_prod(model_base_prx.layers[-1].output_shape[1:]))
model_projector       = tf.keras.Sequential([tf.keras.Input(shape=(projector_input_shape)),
                                             tf.keras.layers.Dense(512, activation = 'relu'),
                                             tf.keras.layers.Dense(128, activation = 'relu')])


# Now we create the model_fsp and model_prx.
inputs_fsp  = tf.keras.Input(shape=(res, res, 3))
x_fsp       = model_base_fsp(inputs_fsp)
x_fsp       = tf.keras.layers.Flatten()(x_fsp)
x_fsp       = tf.keras.layers.BatchNormalization()(x_fsp)
outputs_fsp = tf.keras.layers.Dense(10, activation='softmax')(x_fsp) # ten outputs
model_fsp   = tf.keras.Model(inputs_fsp, outputs_fsp)


inputs_prx  = tf.keras.Input(shape=(res, res, 3))
x_prx       = model_base_prx(inputs_prx)
x_prx       = tf.keras.layers.Flatten()(x_prx)
x_prx       = tf.keras.layers.BatchNormalization()(x_prx)
outputs_prx = model_projector(x_prx) # there are no actual outputs but the last layer of projector
model_prx   = tf.keras.Model(inputs_prx, outputs_prx)


In [ ]:
#@title Some Useful Functions
'''
Abbreviations:
    datain: Input data
    ind   : Index
    tf    : TensorFlow
'''

# Function definition for training a model using the SimCLR approach
def fun_train_simclr(model, dataset, fun_augment_01, fun_augment_02,
                     epochs=100, verbose=1, patience=3, learning_rate=0.001):

    # Determine the output size of the model's last layer
    z_size = model.layers[-1].weights[-1].shape[0]
    # Initialize a list to keep track of the loss values for each epoch
    loss = []

    # Initialize the optimizer with the specified learning rate
    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)

    # Loop through each epoch
    for epoch in range(epochs):
        # Initialize a variable to keep track of the running loss for the current epoch
        loss_running = 0

        # Initialize a progress bar using tqdm to visualize training progress
        pbar = tqdm(dataset,
                    desc  = f"SimCLR Training: {epoch + 1:03d}/{epochs}", # Description shown in the progress bar
                    ncols = 125, # Width of the progress bar
                    leave = True) # Whether the progress bar should remain after completion

        # Iterate over each batch in the dataset
        for batch_num, batch in enumerate(pbar, 1):
            if isinstance(batch, tuple):
                batch_in = batch[0] # Get the input data from the batch
            else:
                batch_in = batch

            # Apply two different augmentations to the input data
            x_tilda_01 = fun_augment_01(batch_in)
            x_tilda_02 = fun_augment_02(batch_in)

            # Concatenate the augmented data along the batch axis and reshape it for the model input
            x_tilda = tf.reshape(tf.concat([x_tilda_01, x_tilda_02], axis=0),
                                (x_tilda_01.shape[0] * 2, # Corrected to ensure the correct shape for concatenation
                                 x_tilda_01.shape[1], x_tilda_01.shape[2],
                                 x_tilda_01.shape[3]))

            # Create a dummy output tensor of the same size as the model's output
            z_real = tf.random.uniform((x_tilda.shape[0], z_size))

            # Open a GradientTape to record the operations for automatic differentiation
            with tf.GradientTape() as tape:
                # Pass the concatenated and augmented inputs through the model
                z_estimate = model(x_tilda, training=True)
                # Calculate the batch loss using the custom SimCLR loss function
                loss_batch = fun_simclr_loss(z_real, z_estimate)

            # Calculate the gradients of the loss with respect to the model's trainable variables
            gradients = tape.gradient(loss_batch, model.trainable_variables)

            # Apply the calculated gradients to the model's variables to minimize the loss
            optimizer.apply_gradients(zip(gradients, model.trainable_variables))

            # Update the running loss for the epoch
            loss_running += loss_batch.numpy()  # Ensure loss is a scalar by calling .numpy()

            # If verbose, update the progress bar with the current average loss
            if verbose:
                pbar.set_postfix(loss = f"{loss_running/batch_num:.6f}")

        # Append the average loss for this epoch to the loss history
        loss.append(loss_running / batch_num)

        # Check for early stopping: if there's no improvement in loss for a specified number of epochs, stop training
        if epoch >= patience:
            if loss[-1] > min(loss[:-patience]):
                print("Early stopping due to no improvement in loss.")
                break  # Exit the training loop

    # Return the trained model and the history of loss values
    return model, loss

#SimCLR loss function
@tf.function
def fun_simclr_loss(z_real, z_estimate):
    # The z_real parameter is not used since SimCLR is an unsupervised model.
    # This dummy variable is discarded.
    del z_real

    # Temperature parameter is set to 0.1. This hyperparameter can be tuned
    # for specific problems to control the smoothness of the output distribution.
    toe = .1

    # Calculate the number of projections, which is twice the batch size (2N).
    num = z_estimate.shape[0]

    # Generate two sets of indices for all possible pair combinations within the batch.
    # ind0 is a temporary variable holding the repeated range for generating indices.
    ind0 = tf.repeat(tf.expand_dims(tf.range(0, num), axis=0), num, axis=0)
    # Flatten ind0 to create a single list of indices for the first element of pairs.
    ind1 = tf.reshape(ind0, (num**2, 1))[:, 0]
    # Flatten the transpose of ind0 to create a single list of indices for the second element of pairs.
    ind2 = tf.reshape(tf.transpose(ind0), (num**2, 1))[:, 0]

    # The temporary index tensor is no longer needed after use.
    del ind0

    # Select the projections based on the first set of indices to form the first elements of pairs.
    vector_1 = tf.gather(z_estimate, ind1, axis=0)
    # The first set of indices is no longer needed after use.
    del ind1

    # Select the projections based on the second set of indices to form the second elements of pairs.
    vector_2 = tf.gather(z_estimate, ind2, axis=0)
    # The second set of indices is no longer needed after use.
    del ind2

    # Calculate the cosine similarity between each pair of projections and negate it to prepare for loss calculation.
    s = -tf.reshape(tf.keras.losses.cosine_similarity(vector_1, vector_2, axis=1), (num, num))

    # The vector tensors are no longer needed after computing similarities.
    del vector_1
    del vector_2

    # Calculate the nominator of the contrastive loss function for each pair.
    nom = tf.exp(s / toe)

    # Calculate the denominator of the contrastive loss function by excluding the self-similarity term.
    x1 = tf.exp(s / toe)
    x2 = 1 - tf.eye(num, dtype=tf.float32)
    denom = tf.repeat(tf.expand_dims(tf.math.reduce_sum(x1 * x2, axis=1), axis=1), num, axis=1)

    # The intermediate tensors used for the denominator are no longer needed.
    del s
    del x1
    del x2

    # Calculate the loss `l(i, j)` for each pair of projections using the computed nominator and denominator.
    l = -tf.math.log(nom / denom)

    # Cleanup intermediate tensors to save memory.
    del nom
    del denom

    # Prepare indices to extract the diagonal elements from the loss matrix, which correspond to the actual pair losses.
    ind_2k0 = tf.range(0, num, 2, dtype=tf.int32)  # Even indices
    ind_2k1 = tf.range(1, num, 2, dtype=tf.int32)  # Odd indices

    # Extract and sum the losses for the actual pairs using the prepared indices.
    loss_mat1_1 = tf.gather(l, ind_2k0, axis=0)
    loss_mat1_2 = tf.gather(loss_mat1_1, ind_2k1, axis=1)
    loss_mat1 = tf.linalg.diag_part(loss_mat1_2)

    loss_mat2_1 = tf.gather(l, ind_2k1, axis=0)
    loss_mat2_2 = tf.gather(loss_mat2_1, ind_2k0, axis=1)
    loss_mat2 = tf.linalg.diag_part(loss_mat2_2)

    # After extracting the necessary elements, the large loss tensor can be discarded.
    del l

    # Combine the individual loss components into a single tensor.
    loss_mat = loss_mat1 + loss_mat2

    # Compute the final loss by taking the sum over all individual losses and normalizing by the number of pairs.
    L = tf.math.reduce_sum(loss_mat) / num


    # Return the final computed loss.
    return L

# Function to compile a model with specified learning rate
def fun_model_compile(model, learning_rate, loss = 'categorical_crossentropy', metrics = [['accuracy']] ):
    # Compiles the model with Adam optimizer, categorical crossentropy as loss, and tracks accuracy
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  loss=loss,
                  metrics=metrics)
    return model

# Function to print the summary of the model
def fun_model_summary(model):
    model.summary()  # Prints the summary of the model
    print('\n')  # Prints a newline for readability

# Function to define callbacks for training
def fun_model_callbacks():
    # Early stopping callback to stop training when val_loss doesn't improve
    cb_early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    # Reduce learning rate callback when val_loss plateaus
    cb_reduce_lr      = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=0.00001)
    return [cb_early_stopping, cb_reduce_lr]

# Function to train the models other than SimCLR with given dataset, epochs, and callbacks
def fun_model_train(model, dataset_tr, dataset_vl, epochs, callbacks):
    model.fit(dataset_tr,  # Training data
              epochs          = epochs,  # Number of epochs to train for
              verbose         = 1,  # Verbosity mode
              shuffle         = True,  # Whether to shuffle the training data
              validation_data = dataset_vl,  # Validation data
              callbacks       = callbacks)  # Callbacks for training
    return model

# Function to evaluate the model with a given dataset
def fun_model_evaluate(model, data):
    _, acc = model.evaluate(data, verbose = 3)
    return [acc]

# Function to compare the weights of base and trained models in fsp and prx tasks
def fun_control(model_base_fsp, model_base_prx, model_fsp, model_prx):
    print('\n')
    # Print examples of weights from the 3rd layer of fully supervised base model
    print('Example of weights in the 3rd  layer of model_base_fsp: ', model_base_fsp.layers[2].weights[0][0][0][0][:2])
    # Print examples of weights from the 3rd layer of pretext base model
    print('Example of weights in the 3rd  layer of model_base_prx: ', model_base_prx.layers[2].weights[0][0][0][0][:2])
    # Print examples of weights from the last layer of fully supervised trained model
    print('Example of weights in the last layer of model_fsp     : ', model_fsp.layers[-1].weights[0][:1][0][:2])
    # Print examples of weights from the last layer of pretext trained model
    print('Example of weights in the last layer of model_prx     : ', model_prx.layers[-1].weights[0][:1][0][:2])


In [ ]:
#@title Training Models
# Capture the start time
t0 = time.time()

# Initiate a result dictionary
results = {}

'''
Initialize transfer learning for model_fsp.
'''

print('\nInitializing transfer learning for model_fsp\n')

# Freeze the base model's layers to prevent updates during the first phase of training.
model_base_fsp.trainable = False

# Freeze the batch norm layer to have its initial parameters (zeros and ones) intact.
model_fsp.layers[-2]     = False

# Compile the model with specified learning rate for transfer learning phase.
model_fsp = fun_model_compile(model_fsp, lr_fsp_trf)

# Store the initial parameters of the last layer of model_fsp for comparison later.
outputs_fsp_initial_parameters = copy.deepcopy(model_fsp.layers[-1].weights)

# Display the model's structure.
fun_model_summary(model_fsp)

# Prepare callbacks for early stopping and learning rate adjustment.
callbacks = fun_model_callbacks()

# Start training the model with specified datasets and number of epochs.
model_fsp = fun_model_train(model_fsp, dataset_tr_fsp, dataset_te, epoch_fsp_trf, callbacks)

# Evaluate the model with a specified dataset.
results ['fsp_trf'] = fun_model_evaluate(model_fsp, dataset_te)

# Check and display changes in parameters after training.
fun_control(model_base_fsp, model_base_prx, model_fsp, model_prx)



'''
Switch to fine-tuning phase for model_fsp.
'''

print('\nStarting fine-tuning for model_fsp\n')

# Unfreeze the base model's layers for fine-tuning.
model_base_fsp.trainable = True

# Unfreeze the batch norm layer.
model_fsp.layers[-2]     = True

# Recompile the model with a new learning rate for fine-tuning.
model_fsp = fun_model_compile(model_fsp, lr_fsp_fnt)

# Display updated model structure.
fun_model_summary(model_fsp)

# Reinitialize callbacks for this phase.
callbacks = fun_model_callbacks()

# Proceed with fine-tuning training.
model_fsp = fun_model_train(model_fsp, dataset_tr_fsp, dataset_te, epoch_fsp_fnt, callbacks)

# Evaluate the model with a specified dataset.
results ['fsp_fnt'] = fun_model_evaluate(model_fsp, dataset_te)

# Monitor changes in model parameters post fine-tuning.
fun_control(model_base_fsp, model_base_prx, model_fsp, model_prx)



'''
Initialize transfer learning for model_prx.
'''

print('\nInitializing transfer learning for model_prx\n')

# Freeze the base model's layers for transfer learning phase.
model_base_prx.trainable = False

# Freeze the batch norm layer to have its initial parameters (zeros and ones) intact.
model_prx.layers[-2]     = False

# Show the model's structure.
fun_model_summary(model_prx)

# Begin training with specified configurations.
model_prx, _ = fun_train_simclr(model_prx,
                                dataset_tr_prx,
                                fun_augment_01,
                                fun_augment_02,
                                epochs     = epoch_prx_trf,
                                verbose    = 1,
                                patience   = 1,
                                learning_rate = lr_prx_trf)

# Evaluate changes in parameters after training.
fun_control(model_base_fsp, model_base_prx, model_fsp, model_prx)



'''
Switch to fine-tuning phase for model_prx.
'''

print('\nStarting fine-tuning for model_prx\n')

# Unfreeze the base model's layers for fine-tuning adjustments.
model_base_prx.trainable = True

# Unfreeze the batch norm layer.
model_prx.layers[-2]     = True

# Begin training with specified configurations.
model_prx, _ = fun_train_simclr(model_prx,
                                dataset_tr_prx,
                                fun_augment_01,
                                fun_augment_02,
                                epochs     = epoch_prx_fnt,
                                verbose    = 1,
                                patience   = 1,
                                learning_rate = lr_prx_fnt)

# Evaluate changes in parameters after training.
fun_control(model_base_fsp, model_base_prx, model_fsp, model_prx)



'''
Construct and prepare model_dwm for transfer learning.
'''
# Constructing model_dwm utilizing the pre-trained model_base_prx, incorporating
# a dense layer similar to that in model_fsp.
inputs_dwm = tf.keras.Input(shape=(res, res, 3))
x_dwm = model_base_prx(inputs_dwm)  # Use the trained base model.
x_dwm = tf.keras.layers.Flatten()(x_dwm)
x_dwm = tf.keras.layers.BatchNormalization()(x_dwm)
outputs_dwm = tf.keras.layers.Dense(10, activation='softmax')(x_dwm)  # Set for 10 output classes.
model_dwm = tf.keras.Model(inputs_dwm, outputs_dwm)


# For equitable comparison between fsp and dwm, we align the parameters of dwm's
# final dense layer with outputs_fsp_initial_parameters.
# Additionally, we transfer and freeze the pre-trained weights of the preceding
# dense layer from prx into dwm's structure to leverage their learned
# representations.
# Set the initial parameters for fair comparison and incorporate pre-trained layers.
model_dwm.layers[-1].set_weights(outputs_fsp_initial_parameters)  # Match initial output layer parameters.
model_dwm.layers[-2].set_weights(model_prx.layers[-2].weights)  # Use pre-trained intermediate layer.

'''
Transfer learning for model_dwm.
'''

print('\nInitializing transfer learning for model_dwm\n')

# Freeze base and selected layers for the initial phase.
model_base_prx.trainable = False

# Freeze the batch norm layer to have its initial parameters (zeros and ones) intact.
model_dwm.layers[-2]     = False

# Compile the model for this phase.
model_dwm = fun_model_compile(model_dwm, lr_dwm_trf)

# Display the model structure.
fun_model_summary(model_dwm)

# Set callbacks for training.
callbacks = fun_model_callbacks()

# Begin training the model.
model_dwm = fun_model_train(model_dwm, dataset_tr_dwm, dataset_te, epoch_dwm_trf, callbacks)

# Evaluate the model with a specified dataset.
results ['dwm_trf'] = fun_model_evaluate(model_dwm, dataset_te)

# Check parameter changes after training.
fun_control(model_base_fsp, model_base_prx, model_fsp, model_prx)



'''
Fine-tuning for model_dwm.
'''

print('\nStarting fine-tuning for model_dwm\n')

# Unfreeze layers for fine-tuning.
model_base_prx.trainable = True

# Unfreeze the batch norm layer.
model_dwm.layers[-2]     = True

# Recompile with adjusted learning rate for fine-tuning.
model_dwm = fun_model_compile(model_dwm, lr_dwm_fnt)

# Display the model's updated structure.
fun_model_summary(model_dwm)

# Prepare callbacks for this phase.
callbacks = fun_model_callbacks()

# Proceed to fine-tune the model.
model_dwm = fun_model_train(model_dwm, dataset_tr_dwm, dataset_te, epoch_dwm_fnt, callbacks)

# Evaluate the model with a specified dataset.
results ['dwm_fnt'] = fun_model_evaluate(model_dwm, dataset_te)

# Evaluate parameter changes post fine-tuning.
fun_control(model_base_fsp, model_base_prx, model_fsp, model_prx)

# Display the results.
print('---- Accuracies ----')
print(pd.DataFrame(results).head())
print('\n')
print("Time duration of experiment (sec): ", time.time() - t0)

# Auto runtime disconnection to save CPU/GPU/TPU allocations
from google.colab import runtime
import time
time.sleep(10)
runtime.unassign()


# <font color="#418FDE" size="10" uppercase>**C: SimCLR Contrastive Experiments**</font>
----

In this lecture, you learned to:
* Create SimCLR experiments without pre-trained weights.
* Create SimCLR experiments with pre-trained weights.

In the next Module (Module 5), we will go over "Generative AI, Part 1."